<a href="https://colab.research.google.com/github/darig7w7/ProyectoSaludMental/blob/main/SaludMental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [85]:
import pandas as pd
# Si ambos tienen el acceso directo en "Mi unidad", la ruta será igual para los dos
df = pd.read_csv('/content/drive/MyDrive/EPG-Ciencia de Datos/Programación Ciencia de Datos/survey.csv')

#df.head(100)

In [82]:
# 2. Crear el resumen directo
resumen = pd.concat([
    df.dtypes,           # Tipo de dato técnico
    df.nunique(),        # Cantidad de valores únicos (cardinalidad)
    df.isnull().sum(),   # Cantidad de valores nulos
    (df.isnull().sum() / len(df) * 100).round(2) # Porcentaje de nulos
], axis=1)

# 3. Renombrar columnas para que sea legible
resumen.columns = ['Tipo', 'Únicos', 'Nulos', '% Nulos']

# Mostrar el resumen
print(resumen)

                             Tipo  Únicos  Nulos  % Nulos
Timestamp                  object     884      0     0.00
Age                         int64      53      0     0.00
Gender                     object      49      0     0.00
Country                    object      48      0     0.00
state                      object      45    515    40.91
self_employed              object       2     18     1.43
family_history             object       2      0     0.00
treatment                  object       2      0     0.00
work_interfere             object       4    264    20.97
no_employees               object       6      0     0.00
remote_work                object       2      0     0.00
tech_company               object       2      0     0.00
benefits                   object       3      0     0.00
care_options               object       3      0     0.00
wellness_program           object       3      0     0.00
seek_help                  object       3      0     0.00
anonymity     

In [88]:
resumen = pd.DataFrame({
    'Variable': df.columns,
    'Tipo': df.dtypes.values,
    'Conteos Principales': [str(df[c].value_counts().to_dict())[:100] + "..." for c in df.columns]
})

display(resumen)

summary_df

Timestamp                    category
Age                             int64
Gender                       category
Country                      category
state                        category
self_employed                category
family_history               category
treatment                    category
work_interfere               category
no_employees                 category
remote_work                  category
tech_company                 category
benefits                     category
care_options                 category
wellness_program             category
seek_help                    category
anonymity                    category
leave                        category
mental_health_consequence    category
phys_health_consequence      category
coworkers                    category
supervisor                   category
mental_health_interview      category
phys_health_interview        category
mental_vs_physical           category
obs_consequence              category
comments    

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest

# 1. Vectorizamos la columna Gender (convertimos a números)
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,3))
X = vectorizer.fit_transform(df['Gender'].astype(str))

# 2. Configuramos el modelo de detección de anomalías
# contamination=0.05 significa que esperamos que el 5% de los datos sean "basura"
clf = IsolationForest(contamination=0.05, random_state=42)
predicciones = clf.fit_predict(X.toarray())

# El modelo devuelve: 1 para datos normales, -1 para anomalías
df['Es_Anomalia'] = predicciones

In [45]:
df.head()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments,Es_Anomalia
0,27/08/2014 11:29,37,Female,United States,IL,NaN,No,Yes,Often,Jun-25,...,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN,1
1,27/08/2014 11:29,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Maybe,No,No,No,No,No,Don't know,No,NaN,1
2,27/08/2014 11:29,32,Male,Canada,NaN,NaN,No,No,Rarely,Jun-25,...,No,No,Yes,Yes,Yes,Yes,No,No,NaN,1
3,27/08/2014 11:29,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN,1
4,27/08/2014 11:30,31,Male,United States,TX,NaN,No,No,Never,100-500,...,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN,1


In [73]:
# Ver qué detectó la IA como basura antes de borrar
#print("Ejemplos de datos detectados como anomalías:")
#print(df[df['Es_Anomalia'] == -1]['Gender'].unique())

# Eliminar las filas que son anomalías
df_limpio = df[df['Es_Anomalia'] == 1].copy()

# Borrar la columna auxiliar
df_limpio = df_limpio.drop(columns=['Es_Anomalia'])

print(f"\nFilas originales: {len(df)}")
print(f"Filas después de la limpieza con IA: {len(df_limpio)}")


Filas originales: 1259
Filas después de la limpieza con IA: 1259


In [19]:
print(df.dtypes)

Timestamp                    datetime64[ns]
Age                                   int64
Gender                               object
Country                              object
state                                object
self_employed                        object
family_history                       object
treatment                            object
work_interfere                       object
no_employees                         object
remote_work                          object
tech_company                         object
benefits                             object
care_options                         object
wellness_program                     object
seek_help                            object
anonymity                            object
leave                                object
mental_health_consequence            object
phys_health_consequence              object
coworkers                            object
supervisor                           object
mental_health_interview         

In [34]:
df.head(779)

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments,Gender_Cluster
0,27/08/2014 11:29,37,F,United States,IL,NaN,No,Yes,Often,Jun-25,...,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN,0
1,27/08/2014 11:29,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Maybe,No,No,No,No,No,Don't know,No,NaN,1
2,27/08/2014 11:29,32,F,Canada,NaN,NaN,No,No,Rarely,Jun-25,...,No,No,Yes,Yes,Yes,Yes,No,No,NaN,0
3,27/08/2014 11:29,31,F,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN,0
4,27/08/2014 11:30,31,F,United States,TX,NaN,No,No,Never,100-500,...,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,28/08/2014 12:07,37,F,United Kingdom,NaN,No,Yes,Yes,Rarely,Jun-25,...,Yes,No,Some of them,Yes,No,Maybe,No,Yes,Some of these questions are not really suitabl...,0
775,28/08/2014 12:08,39,F,Canada,NaN,No,No,Yes,Sometimes,100-500,...,Yes,No,Some of them,Yes,No,Maybe,No,Yes,NaN,0
776,28/08/2014 12:09,43,M,United States,VA,No,No,No,Sometimes,Jun-25,...,Maybe,No,No,No,No,Maybe,No,No,NaN,1
777,28/08/2014 12:10,32,M,United Kingdom,NaN,No,No,Yes,Sometimes,More than 1000,...,No,No,Yes,Some of them,Yes,Yes,Yes,Yes,NaN,1


In [67]:
df_limpio.head(954)

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,27/08/2014 11:29,37,Female,United States,IL,NaN,No,Yes,Often,Jun-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,27/08/2014 11:29,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,27/08/2014 11:29,32,Male,Canada,NaN,NaN,No,No,Rarely,Jun-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,27/08/2014 11:29,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,27/08/2014 11:30,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983,29/08/2014 09:01,28,male,United States,NY,No,Yes,Yes,Often,100-500,...,Don't know,Maybe,No,Some of them,Some of them,No,Yes,Don't know,No,NaN
984,29/08/2014 09:01,27,Male,Canada,NaN,No,No,No,Never,26-100,...,Very easy,No,No,Yes,Yes,Maybe,Maybe,Yes,No,NaN
985,29/08/2014 09:02,29,Male,United Kingdom,NaN,No,No,Yes,Sometimes,Jun-25,...,Very easy,No,No,Yes,Yes,No,Yes,Yes,No,NaN
986,29/08/2014 09:02,39,Female,United Kingdom,NaN,No,Yes,No,Rarely,500-1000,...,Somewhat difficult,Yes,Yes,No,No,No,No,No,Yes,NaN
